# Process website_alldata (clean pipeline)

This notebook creates consistent processed outputs for the website dataset.

Outputs (in `Data/output_data`):
- `ncrna_symbol_list.txt`
- `website_sequences.csv` (optional fetch step)
- `website_disease_matrix.csv`
- `website_full_matrix.csv` (if `website_sequences.csv` exists)
- `website_sequences_for_oop.csv` (if `website_sequences.csv` exists)
- `sequences_for_oop.csv` (if `Data/raw/sequences.csv` exists)
- `dinuc_props.csv`
- `do_terms.csv`, `do_edges.csv`, `disease_terms_mapping.csv` (if OBO exists)


In [ ]:
from pathlib import Path
import re
import time
import requests
import pandas as pd

try:
    import obonet
except Exception:
    obonet = None


def find_project_root(marker_rel: Path = Path("Data/raw/website_alldata.csv")) -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / marker_rel).exists():
            return base
    raise FileNotFoundError(f"Could not locate project root containing {marker_rel}")


PROJECT_ROOT = find_project_root()
RAW_WEBSITE = PROJECT_ROOT / "Data/raw/website_alldata.csv"
RAW_V2 = PROJECT_ROOT / "Data/raw/sequences.csv"
DO_OBO = PROJECT_ROOT / "Data/raw/HumanDO.obo"
OUT_DIR = PROJECT_ROOT / "Data/output_data"

SYMBOLS_TXT = OUT_DIR / "ncrna_symbol_list.txt"
WEBSITE_SEQS = OUT_DIR / "website_sequences.csv"
FETCH_REPORT = OUT_DIR / "sequence_fetch_report.csv"
WEBSITE_DISEASE = OUT_DIR / "website_disease_matrix.csv"
WEBSITE_FULL = OUT_DIR / "website_full_matrix.csv"
WEBSITE_OOP = OUT_DIR / "website_sequences_for_oop.csv"
V2_OOP = OUT_DIR / "sequences_for_oop.csv"
DINUC_PROPS = OUT_DIR / "dinuc_props.csv"
DO_TERMS = OUT_DIR / "do_terms.csv"
DO_EDGES = OUT_DIR / "do_edges.csv"
DO_MAP = OUT_DIR / "disease_terms_mapping.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_WEBSITE.exists():
    raise FileNotFoundError(f"Missing input: {RAW_WEBSITE}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Using website raw: {RAW_WEBSITE}")
print(f"Output dir: {OUT_DIR}")


In [ ]:
# 1) Load + filter to human lncRNA only (single source of truth)
raw = pd.read_csv(RAW_WEBSITE)
flt = raw[(raw["Species"] == "Homo sapiens") & (raw["ncRNA Category"] == "LncRNA")].copy()

print("raw rows:", len(raw))
print("filtered rows:", len(flt))
print("unique symbols:", flt["ncRNA Symbol"].nunique())


In [ ]:
# 2) Symbol list for sequence fetching
symbols = sorted(flt["ncRNA Symbol"].dropna().astype(str).unique())
with open(SYMBOLS_TXT, "w") as f:
    for s in symbols:
        f.write(s + "
")
print(f"[saved] {SYMBOLS_TXT} ({len(symbols)} symbols)")


In [ ]:
# 3) Optional: fetch sequences from Ensembl (resume-friendly)
# Set RUN_FETCH=True only when you want to query online services.
RUN_FETCH = False
MAX_SYMBOLS = None      # set int for quick test, e.g. 200
SLEEP_SEC = 0.05
TIMEOUT = 10

ENSEMBL_LOOKUP = "https://rest.ensembl.org/lookup/symbol/homo_sapiens/{symbol}"
ENSEMBL_SEQ = "https://rest.ensembl.org/sequence/id/{ensembl_id}"

if RUN_FETCH:
    session = requests.Session()
    existing = {}
    if WEBSITE_SEQS.exists():
        prev = pd.read_csv(WEBSITE_SEQS, dtype=str)
        existing = {r.ID: r.seqs for r in prev.itertuples() if isinstance(r.seqs, str) and r.seqs.strip()}

    target_symbols = symbols[:MAX_SYMBOLS] if MAX_SYMBOLS else symbols
    rows, report = [], []

    def fetch_from_ensembl(symbol: str):
        try:
            r = session.get(ENSEMBL_LOOKUP.format(symbol=symbol), headers={"Accept": "application/json"}, timeout=TIMEOUT)
            if r.status_code != 200:
                return None, f"lookup_status:{r.status_code}"
            eid = r.json().get("id")
            if not eid:
                return None, "lookup_no_id"
            rs = session.get(ENSEMBL_SEQ.format(ensembl_id=eid), headers={"Accept": "text/plain"}, timeout=TIMEOUT)
            if rs.status_code != 200:
                return None, f"seq_status:{rs.status_code}"
            seq = rs.text.strip()
            return (seq if seq else None), f"ensembl:{eid}"
        except Exception as e:
            return None, f"err:{e}"

    for sym in target_symbols:
        if sym in existing:
            rows.append({"ID": sym, "seqs": existing[sym]})
            report.append({"ID": sym, "status": "cached", "detail": "existing"})
            continue

        seq, detail = fetch_from_ensembl(sym)
        rows.append({"ID": sym, "seqs": seq})
        report.append({"ID": sym, "status": "ok" if seq else "failed", "detail": detail})
        time.sleep(SLEEP_SEC)

    pd.DataFrame(rows, columns=["ID", "seqs"]).to_csv(WEBSITE_SEQS, index=False)
    pd.DataFrame(report).to_csv(FETCH_REPORT, index=False)
    print(f"[saved] {WEBSITE_SEQS} ({len(rows)} rows)")
    print(f"[saved] {FETCH_REPORT}")
else:
    print("RUN_FETCH=False -> skipping online fetch step")


In [ ]:
# 4) Build disease matrix from filtered data (ID + 0/1 disease columns)
flt2 = flt.copy()
flt2["present"] = 1

disease_mat = (
    flt2.pivot_table(
        index="ncRNA Symbol",
        columns="Disease Name",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
    .rename(columns={"ncRNA Symbol": "ID"})
)

disease_mat.to_csv(WEBSITE_DISEASE, index=False)
print(f"[saved] {WEBSITE_DISEASE} shape={disease_mat.shape}")


In [ ]:
# 5) Merge sequences + disease matrix (if sequences file exists)
if WEBSITE_SEQS.exists():
    seqs = pd.read_csv(WEBSITE_SEQS)
    full = disease_mat.merge(seqs, on="ID", how="inner")
    disease_cols = [c for c in full.columns if c not in ("ID", "seqs")]
    full = full[["ID", "seqs", *disease_cols]]
    full.to_csv(WEBSITE_FULL, index=False)
    print(f"[saved] {WEBSITE_FULL} shape={full.shape}")

    oop = seqs.rename(columns={"ID": "id", "seqs": "seq"})[["id", "seq"]]
    oop.to_csv(WEBSITE_OOP, index=False)
    print(f"[saved] {WEBSITE_OOP} shape={oop.shape}")
else:
    print(f"[skip] {WEBSITE_SEQS} not found; skipping merge + website OOP export")


In [ ]:
# 6) Optional: V2 sequences to oop format
if RAW_V2.exists():
    v2 = pd.read_csv(RAW_V2)
    v2_oop = v2.rename(columns={"ID": "id", "seqs": "seq"})[["id", "seq"]]
    v2_oop.to_csv(V2_OOP, index=False)
    print(f"[saved] {V2_OOP} shape={v2_oop.shape}")
else:
    print(f"[skip] {RAW_V2} not found")


In [ ]:
# 7) Build dinucleotide properties from available sequence data (RNA alphabet A,C,G,U)
seq_sources = [WEBSITE_SEQS, WEBSITE_OOP, V2_OOP, RAW_V2]
seqs = []
for src in seq_sources:
    if not src.exists():
        continue
    s = pd.read_csv(src, dtype=str)
    s.columns = [c.lower() for c in s.columns]
    col = "seqs" if "seqs" in s.columns else ("seq" if "seq" in s.columns else None)
    if not col:
        continue
    seqs = s[col].dropna().astype(str).tolist()
    print(f"Using sequences from {src} ({len(seqs)} entries)")
    break

if seqs:
    from collections import Counter

    counts = Counter()
    total = 0
    for seq in seqs:
        x = "".join(seq.split()).upper().replace("T", "U")
        for i in range(len(x) - 1):
            d = x[i:i+2]
            if len(d) == 2 and set(d) <= set("ACGU"):
                counts[d] += 1
                total += 1

    rows = []
    for d in [a + b for a in "ACGU" for b in "ACGU"]:
        c = counts[d]
        rows.append({"dinuc": d, "count": c, "freq": (c / total if total else 0.0)})

    props = pd.DataFrame(rows)
    props.to_csv(DINUC_PROPS, index=False)
    print(f"[saved] {DINUC_PROPS} shape={props.shape}")
else:
    print("[skip] no sequence source found for dinuc props")


In [ ]:
# 8) Optional: build Disease Ontology helpers using obonet
if DO_OBO.exists() and obonet is not None:
    G = obonet.read_obo(DO_OBO)

    edges = []
    for u, v, k, d in G.edges(keys=True, data=True):
        rel = d.get("relation") or k
        if rel == "is_a":
            edges.append({"child": u, "parent": v})
    edges_df = pd.DataFrame(edges).drop_duplicates()

    rows = []
    for doid, data in G.nodes(data=True):
        if data.get("is_obsolete") == "true":
            continue
        syns = data.get("synonym", [])
        if isinstance(syns, str):
            syns = [syns]
        clean_syns = [s.split('"')[1] if '"' in s else s for s in syns]
        rows.append({"doid": doid, "name": data.get("name", ""), "synonyms": ";".join(clean_syns)})

    terms_df = pd.DataFrame(rows)
    terms_df.to_csv(DO_TERMS, index=False)
    edges_df.to_csv(DO_EDGES, index=False)
    print(f"[saved] {DO_TERMS} shape={terms_df.shape}")
    print(f"[saved] {DO_EDGES} shape={edges_df.shape}")

    # disease -> DO term mapping from disease columns
    diseases = [c for c in disease_mat.columns if c != "ID"]

    def norm_text(s: str) -> str:
        s = str(s or "").strip().lower()
        s = re.sub(r"[\(\)\[\]\{\},;:/\\-\+_]", " ", s)
        s = re.sub(r"\s+", " ", s).strip()
        return s

    t = terms_df.fillna("").copy()
    t["name_norm"] = t["name"].map(norm_text)
    t["syn_norms"] = t["synonyms"].map(lambda s: [norm_text(x) for x in str(s).split(";") if norm_text(x)])

    name_to_doid, syn_to_doid = {}, {}
    for _, row in t.iterrows():
        doid = row["doid"]
        nn = row["name_norm"]
        if nn and nn not in name_to_doid:
            name_to_doid[nn] = doid
        for sn in row["syn_norms"]:
            if sn and sn not in syn_to_doid:
                syn_to_doid[sn] = doid

    mapping = []
    for d in diseases:
        dn = norm_text(d)
        term = name_to_doid.get(dn, "") or syn_to_doid.get(dn, "")
        mapping.append({"disease": d, "term": term})

    map_df = pd.DataFrame(mapping)
    map_df.to_csv(DO_MAP, index=False)
    print(f"[saved] {DO_MAP} shape={map_df.shape} (mapped={(map_df['term'] != '').sum()})")
else:
    if not DO_OBO.exists():
        print(f"[skip] {DO_OBO} not found")
    if obonet is None:
        print("[skip] obonet not installed")


In [ ]:
# 9) Validation summary
for p in [SYMBOLS_TXT, WEBSITE_SEQS, FETCH_REPORT, WEBSITE_DISEASE, WEBSITE_FULL, WEBSITE_OOP, V2_OOP, DINUC_PROPS, DO_TERMS, DO_EDGES, DO_MAP]:
    if p.exists():
        try:
            d = pd.read_csv(p)
            print(f"{p}: shape={d.shape}")
        except Exception:
            print(f"{p}: exists (non-tabular)")
    else:
        print(f"{p}: missing")

if WEBSITE_SEQS.exists():
    s = pd.read_csv(WEBSITE_SEQS)
    if "seqs" in s.columns:
        print("website_sequences missing seq count:", s["seqs"].isna().sum())
        print("website_sequences unique IDs:", s["ID"].nunique() if "ID" in s.columns else "n/a")

if WEBSITE_FULL.exists() and WEBSITE_DISEASE.exists():
    f = pd.read_csv(WEBSITE_FULL)
    y = pd.read_csv(WEBSITE_DISEASE)
    print("full matrix has seqs col:", "seqs" in f.columns)
    print("disease matrix unique IDs:", y["ID"].nunique() if "ID" in y.columns else "n/a")
    print("full matrix unique IDs:", f["ID"].nunique() if "ID" in f.columns else "n/a")
